In [30]:
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src/')
import features.prepare_data as pp
import models.ptype_run_preds as rp
import models.score_classifier as score
import report.ptype_visualize as viz
import pandas as pd
import pickle
import os
import boto3
from datetime import datetime
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
import hydra
from omegaconf import DictConfig
from utils.logs import get_logger
import yaml
import data.s3_download_sso as sso_load
import data.clean_ceo_summary as ceo_clean

In [31]:
config_path='../params.yaml'
with open(config_path) as conf_file:
    config = yaml.safe_load(conf_file)
logger = get_logger('DATA_LOAD', log_level=config['base']['log_level'])

In [32]:
@hydra.main(config_path='params.yaml')
def main(config: DictConfig) -> None:
    print(config.data_load.sso_profile_name)

/tmp/ipykernel_492497/4294687443.py:1: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path='params.yaml')


In [33]:
config['data_condition']['train_filter_features']

{'Use?': 'yes', 'Classes': 'multi'}

In [34]:
sso_load.data_download(config_path)
ceo_summary = ceo_clean.import_ceo_summary(config_path)
ceo_batch_list = ceo_clean.id_ceo_batches(config_path, ceo_summary)

2023-10-16 15:33:04,931 — DATA_DOWNLOAD — INFO — No new data downloaded
2023-10-16 15:33:04,956 — CEO_SUMMARY_LOAD — INFO — Summary data loaded


In [35]:
ceo_batch_list

['v08', 'v15', 'v18', 'v19']

In [36]:
X, y = pp.create_xy(ceo_batch_list, 
                    classes='multi', 
                    drop_feats=False,
                    data_dir =config['data_load']['ceo_survey_directory'], 
                    verbose=False)

37 plots labeled unknown were dropped from v08.
2 plots labeled unknown were dropped from v15.
49 plots labeled unknown were dropped from v18.
23 plots labeled unknown were dropped from v19.
514 plots had no cloud free imagery and will be removed.
Training data includes 0 plots.


0it [00:00, ?it/s]

Class count {}


In [23]:
os.path.exists('../data/')

True